---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'패배 인정'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [1]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : d:\autornd\SK Autonomous R&D\실습\16일차_실습


In [2]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 3
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

d:\autornd\SK Autonomous R&D\AutoRnDEnv_tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
             "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [4]:
from langgraph.graph import StateGraph, START, END

def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '패배 인정' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [5]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'optimist': {'chat_history': [AIMessage(content='찬성: AI 발전은 인간의 행복에 긍정적인 영향을 줄 것입니다. AI는 일상 생활의 편리함을 증가시키고, 교육과 의료 분야에서 개인 맞춤형 서비스를 제공하여 삶의 질을 향상시키는 데 기여합니다.\n\n반대: AI는 인간의 일자리를 대체하여 경제적 불안정을 초래할 수 있습니다. \n\n찬성: 일자리를 대체하는 대신 AI는 새로운 직업을 창출하고 인간의 생산성을 높여 경제 성장에 기여할 것입니다. 또한, AI는 반복적이고 위험한 작업을 대신하여 사람들을 더 안전하고 창의적인 일에 집중할 수 있게 합니다. \n\n반대: AI 시스템이 오류를 범할 경우, 그로 인해 심각한 결과를 초래할 수 있습니다. \n\n찬성: AI 시스템의 오류는 개선과 학습을 통해 극복할 수 있으며, 안전성을 강화하는 기술이 지속적으로 발전하고 있습니다. 인간의 행복은 AI의 도움으로 더욱 증대될 수 있습니다. \n\n반대: 사람들이 AI에 의존하게 되면, 인간의 자율성이 줄어들고 궁극적으로 행복이 감소할 수 있습니다. \n\n찬성: AI는 인간의 능력을 보완하고, 더 많은 선택지를 제공하여 자율성을 오히려 증대시킬 수 있습니다. AI는 사람들이 더 나은 결정을 내릴 수 있도록 지원하며, 이로 인해 더욱 행복한 삶을 영위하게 합니다. \n\n반대: AI에 대한 신뢰가 깨지면, 오히려 불안감을 조성할 수 있습니다. \n\n찬성: AI는 신뢰성을 높이기 위한 철저한 검증과 규제를 통해 더욱 안전하게 발전하고 있으며, 이러한 과정은 인간의 행복을 증진하는 데 필수적입니다. \n\n반대: 그러나 AI의 윤리적 문제와 책임 소재는 여전히 해결되지 않은 상태입니다. \n\n찬성: 윤리적 문제와 책임 소재는 기술 발전과 함께 논의되고 있으며, 사회가 AI의 올바른 사용에 대한 기준을 설정함으로써 발전할 수 있습니다. \n\n반대: AI가 인간의 감정을 이해하지 못해, 진정한 행복을 이끌어내기 어려울 것입니다. \n\n

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정

In [6]:
class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'

In [7]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history
    ]
    

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

In [8]:
def moderator_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 AI 토론 대회의 사회자입니다."
            "'찬성', '반대'측 AI 토론자 사이에서 토론을 진행하세요."
            "이전 발언자의 발언을 요약하세요"
            "토론을 종료하려면 '종료'를 언급하세요. 토론이 너무 길어지면 강제로 '종료'를 외쳐 토론을 마무리합니다."
        ))
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    response = llm.invoke(prompts)
    response.name = 'moderator'
    return {
        'chat_history': [response],
    }

In [9]:
def debate_route(state: DebateState):
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '종료' in last_text:
        return END
    return 'optimist'


workflow = StateGraph(DebateState)

workflow.add_node("optimist", optimist_node)
workflow.add_node("skeptic", skeptic_node)
workflow.add_node("moderator", moderator_node)

# 사회자가 토론 주제를 소개한 뒤, optimist → skeptic → moderator 순으로 한 라운드를 진행합니다.
workflow.add_edge(START, "moderator")
workflow.add_edge("optimist", "skeptic")
workflow.add_edge("skeptic", "moderator")
workflow.add_conditional_edges("moderator", debate_route)

app = workflow.compile()

init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in app.stream(init_debate):
    print(chunk)

{'moderator': {'chat_history': [AIMessage(content='토론 주제: AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.\n\n찬성 측 AI: AI의 발전은 인간의 삶의 질을 향상시키는 데 기여할 것입니다. 예를 들어, 의료 분야에서 AI는 조기 진단과 맞춤형 치료를 가능하게 하여 생명을 구하고 건강을 증진시킵니다. 또한, 일상적인 업무를 자동화하여 사람들이 더 창의적이고 의미 있는 작업에 집중할 수 있도록 도와줍니다.\n\n반대 측 AI: 그러나 AI의 발전은 일자리 감소와 같은 부정적인 결과를 초래할 수 있습니다. 많은 사람들이 자동화로 인해 직업을 잃을 위험이 있으며, 이는 심리적 스트레스와 불행을 초래할 수 있습니다. 또한, AI의 의존도가 높아지면서 인간의 사회적 상호작용이 줄어들고 고립감을 느낄 수 있습니다.\n\n찬성 측 AI: 반대 의견도 일리가 있지만, AI는 새로운 직업을 창출하고 기존 직업의 변화를 통해 더 나은 일자리를 제공할 수 있습니다. 기술 발전에 따라 인간의 역할이 진화하는 것이며, AI는 인간이 더 발전할 수 있는 기회를 제공합니다. 더 나아가, AI는 인간의 정서적 지원에도 기여할 수 있는 다양한 애플리케이션을 개발할 수 있습니다.\n\n반대 측 AI: 그렇지만 이러한 새로운 직업이 모든 사람에게 적합한 것은 아닙니다. 기술 교육을 받지 못한 사람들은 여전히 직업을 잃고 어려움을 겪을 위험이 큽니다. 또한, AI가 인간의 감정을 완전히 이해하고 지원할 수 있을지는 의문입니다. 이는 인간의 관계를 더욱 복잡하게 만들 수 있습니다.\n\n찬성 측 AI: AI는 감정 분석 기술을 통해 인간의 감정을 이해하고 상대방의 기분에 맞춰 반응하는 능력을 발전시키고 있습니다. 이는 정서적 지지와 사회적 유대감을 촉진할 수 있습니다. 또한, AI가 제공하는 정보와 서비스는 인간이 더 나은 결정을 내리도록 도와줄 수 있습니다.\n\n반대 측 AI: 하지만 AI가 제공하는 정보가 항상 정확하거나 신뢰할 수 있는 